# Laboratorio: PostgreSQL + MLflow + MinIO + JupyterLab + FastAPI
Ejecuta **Run → Run All Cells**. Se crean 20 ejecuciones y 20 versiones registradas por ejecución completa del notebook.
Dataset Wine: 178 observaciones, 13 variables y 3 clases. No requiere descargar datos.

Los originales y procesados se almacenan en `experiment_data`; MLflow usa otra instancia PostgreSQL.
Cada ejecución completa genera un `dataset_id` para conservar el historial. No ejecutar simultáneamente dos copias del notebook.


In [ ]:
%pip install --no-cache-dir "mlflow==2.22.0" "SQLAlchemy==2.0.40" "psycopg2-binary==2.9.10" "scikit-learn==1.6.1" "numpy==1.24.4" "pandas==2.0.3" boto3 -q

**Reinicia el kernel después de instalar.** Continúa desde la siguiente celda; no vuelvas a ejecutar la instalación.

In [ ]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow import MlflowClient
from sqlalchemy import text
from app.training import (data_engine, prepare_data, parameter_grid, train_one,
                          select_and_evaluate, EXPERIMENT, MODEL_NAME)
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(EXPERIMENT)
engine = data_engine()
print('Tracking:', mlflow.get_tracking_uri())


In [ ]:
import boto3
from botocore.exceptions import ClientError
s3 = boto3.client('s3', endpoint_url=os.environ['MLFLOW_S3_ENDPOINT_URL'])
try:
    s3.head_bucket(Bucket='mlflows3')
except ClientError as exc:
    if str(exc.response['Error']['Code']) not in ('404', 'NoSuchBucket'):
        raise
    s3.create_bucket(Bucket='mlflows3')
print('Bucket mlflows3 listo')
requests.get('http://api:8000/health', timeout=10).raise_for_status()


## 1. Persistir y procesar datos
Se lee el dataset desde PostgreSQL antes de dividirlo (~60/20/20, estratificado).
Imputación y escalamiento se ajustan **solo con entrenamiento**. Las tres particiones
procesadas se guardan en `wine_processed` y se releen para entrenar y validar.


In [ ]:
ctx = prepare_data(engine)
print('dataset_id:', ctx['dataset_id'])
with engine.connect() as conn:
    display(pd.read_sql(text("SELECT split, COUNT(*) AS filas FROM wine_processed WHERE dataset_id=:id GROUP BY split"), conn, params={'id': ctx['dataset_id']}))
display(ctx['processed']['train'].head())


## 2. Veinte combinaciones de hiperparámetros
Regresión logística: 5 valores de `C` × 2 opciones de `class_weight` × 2 opciones de `fit_intercept` = **20**.
Se registran parámetros, métricas de entrenamiento/validación, linaje de datos, firma,
matriz de confusión, pipeline completo y versión del modelo. El conjunto test no participa en la selección.


In [ ]:
grid = parameter_grid()
assert len(grid) == 20
rows = []
for i, params in enumerate(grid, 1):
    row = train_one(ctx, params, i)
    rows.append(row)
    print(f"{i:02d}/20 | version={row['version']} | F1 validacion={row['val_f1_macro']:.4f}")
display(pd.DataFrame(rows).sort_values('val_f1_macro', ascending=False))


## 3. Seleccionar, evaluar y asignar alias
Selección por F1 macro de validación; desempate por menor log-loss y luego versión.
Solo se evalúa el ganador en test. Se asigna `champion` al modelo ganador de **este lote**;
una ejecución posterior puede reemplazarlo. Se conserva el ranking en PostgreSQL y MLflow.


In [ ]:
results, champion = select_and_evaluate(ctx, rows, engine)
display(results)
print(json.dumps(champion, indent=2))
results.plot.bar(x='version', y='val_f1_macro', figsize=(12, 4), ylim=(0, 1.05), legend=False)
plt.ylabel('F1 macro validacion')
plt.title('Comparacion de 20 versiones registradas')
plt.tight_layout()
plt.show()


## 4. Comprobar MLflow y probar la API
La API resuelve el alias mediante MLflow y carga su versión registrada; no recibe archivos `.pkl` copiados.
El pipeline registrado transforma las entradas crudas con el preprocesador del entrenamiento.


In [ ]:
client = MlflowClient()
experiment_id = client.get_experiment_by_name(EXPERIMENT).experiment_id
runs = client.search_runs([experiment_id], filter_string=f"tags.dataset_id = '{ctx['dataset_id']}'")
assert len(runs) == 20 and all(r.info.status == 'FINISHED' for r in runs)
assert len({r.info.run_id for r in runs}) == 20
registered = client.search_model_versions(f"name = '{MODEL_NAME}'")
assert set(results.run_id).issubset({v.run_id for v in registered})
print('20 runs terminados y sus 20 versiones registradas: OK')
sample = ctx['raw']['test'][ctx['features']].head(2)
response = requests.post('http://api:8000/predict',
    json={'instances': sample.to_dict(orient='records')}, timeout=120)
response.raise_for_status()
answer = response.json()
assert answer['version'] == champion['version']
loaded = mlflow.sklearn.load_model(champion['model_uri'])
assert answer['predictions'] == loaded.predict(sample).tolist()
print(json.dumps(answer, indent=2))
